# 03 · Targeting recommendation

Notebooks 01–02 showed that the men's e-mail beats the women's e-mail overall and works in every segment, while the women's e-mail only works for some customers. The business question is now:

> **Who should get which e-mail — and is targeting worth it compared with simply sending the men's e-mail to everyone?**

**Method.** Because customers were randomized, the value of *any* sending rule can be estimated directly from the experiment with inverse-probability weighting (IPW): keep the customers who happened to receive the e-mail the rule would choose and weight them by 1 / P(receiving it). No predictive model is needed (see `src/policy.py`).

**Outcome:** profit per customer = margin × spend − e-mail cost. Both are assumptions, set below and varied in section 4.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import data, policy as pol

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

df = data.load(data.download())
df["past_purchase"] = np.select(
    [(df["mens"] == 1) & (df["womens"] == 1), df["mens"] == 1, df["womens"] == 1],
    ["Both", "Mens only", "Womens only"], default="Neither")
df["customer_type"] = np.where(df["newbie"] == 1, "New", "Existing")

# --- Business assumptions (not in the data; change them and rerun) ---
MARGIN = 0.30          # gross margin on merchandise revenue
COST_PER_EMAIL = 0.01  # $ per e-mail sent (creative + delivery)
CONTROL = data.CONTROL

## 1 · Candidate sending rules

| Rule | Logic |
|---|---|
| No e-mail | baseline |
| Women's e-mail to all | |
| Men's e-mail to all | simplest option, best on average in notebook 01 |
| Match past purchase | women-only buyers get the women's e-mail, everyone else the men's |
| Learned rule (past purchase × customer type) | for each of the cells, pick the arm with the highest profit in the data |

In [ ]:
CELL_COLS = ["past_purchase", "customer_type"]
learned_in_sample = pol.learn_cell_rule(df, CELL_COLS, MARGIN, COST_PER_EMAIL, CONTROL)
pd.Series(learned_in_sample, name="chosen arm").to_frame()

In [ ]:
policies = {
    "No e-mail": pol.everyone(CONTROL),
    "Women's e-mail to all": pol.everyone("Womens E-Mail"),
    "Men's e-mail to all": pol.everyone("Mens E-Mail"),
    "Match past purchase": pol.match_past_purchase,
    "Learned rule (in-sample)": pol.apply_cell_rule(learned_in_sample, CELL_COLS, "Mens E-Mail"),
}

values, contribs = {}, {}
for name, p in policies.items():
    values[name], contribs[name] = pol.policy_value(df, p, MARGIN, COST_PER_EMAIL, CONTROL)

rows = []
for name in policies:
    d_none, lo_none, hi_none = pol.paired_bootstrap_diff(contribs[name], contribs["No e-mail"])
    d_men, lo_men, hi_men = pol.paired_bootstrap_diff(contribs[name], contribs["Men's e-mail to all"])
    rows.append({"rule": name, "profit_per_customer": values[name],
                 "vs_no_email_per_1000": d_none * 1000, "ci_low": lo_none * 1000, "ci_high": hi_none * 1000,
                 "vs_mens_to_all_per_1000": d_men * 1000, "ci_low_vs_mens": lo_men * 1000, "ci_high_vs_mens": hi_men * 1000})
results = pd.DataFrame(rows)
results

**How to read this:** `vs_no_email_per_1000` is the incremental profit per 1,000 customers compared with sending nothing; `vs_mens_to_all_per_1000` compares each rule with the simple "men's e-mail to everyone" rule. A targeted rule is only worth the extra complexity if its CI vs. men's-to-all lies clearly above 0.

In [ ]:
plot_df = results[results["rule"] != "No e-mail"]
fig, ax = plt.subplots(figsize=(8, 3.5))
y = np.arange(len(plot_df))
ax.errorbar(plot_df["vs_no_email_per_1000"], y,
            xerr=[plot_df["vs_no_email_per_1000"] - plot_df["ci_low"], plot_df["ci_high"] - plot_df["vs_no_email_per_1000"]],
            fmt="o", capsize=4)
ax.axvline(0, color="grey", ls="--", lw=1)
ax.set_yticks(y, plot_df["rule"])
ax.set_xlabel(f"Incremental profit per 1,000 customers vs. no e-mail ($)\nmargin {MARGIN:.0%}, cost ${COST_PER_EMAIL:.2f}/e-mail")
ax.set_title("Value of each sending rule (95% bootstrap CI)")
plt.tight_layout()
plt.show()

## 2 · Honest evaluation of the learned rule

The in-sample learned rule is **optimistic by construction**: it picks, in each cell, whichever arm happened to do best — including by chance. To get an honest estimate, the rule is learned on a random half of the customers and evaluated on the other half, repeated 200 times.

In [ ]:
rng = np.random.default_rng(2026)
gains = []
for _ in range(200):
    train_mask = rng.random(len(df)) < 0.5
    train, test = df[train_mask], df[~train_mask]
    rule = pol.learn_cell_rule(train, CELL_COLS, MARGIN, COST_PER_EMAIL, CONTROL)
    v_learned, _ = pol.policy_value(test, pol.apply_cell_rule(rule, CELL_COLS, "Mens E-Mail"), MARGIN, COST_PER_EMAIL, CONTROL)
    v_mens, _ = pol.policy_value(test, pol.everyone("Mens E-Mail"), MARGIN, COST_PER_EMAIL, CONTROL)
    gains.append((v_learned - v_mens) * 1000)
gains = np.array(gains)

print(f"Learned rule vs. men's-to-all on held-out data, per 1,000 customers:")
print(f"  mean = {gains.mean():+.1f} $,  middle 95% of splits = [{np.quantile(gains, .025):+.1f}, {np.quantile(gains, .975):+.1f}] $")
print(f"  learned rule better in {np.mean(gains > 0):.0%} of splits")

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(gains, bins=30)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Held-out profit gain vs. men's-to-all ($ per 1,000 customers)")
ax.set_ylabel("Number of splits")
ax.set_title("Does the learned targeting rule generalize?")
plt.tight_layout()
plt.show()

## 3 · Is sending worth it at all? Break-even e-mail cost

An e-mail pays off when margin × spend lift > cost per e-mail. The table shows the maximum cost per e-mail at which sending still breaks even, for different margins.

In [ ]:
ctrl_spend = df.loc[df["segment"] == CONTROL, "spend"].mean()
lifts = {arm: df.loc[df["segment"] == arm, "spend"].mean() - ctrl_spend for arm in data.TREATMENTS}
breakeven = pd.DataFrame({
    "margin": [0.2, 0.3, 0.4, 0.5],
    **{f"max cost per e-mail, {arm} ($)": [m * lifts[arm] for m in [0.2, 0.3, 0.4, 0.5]] for arm in data.TREATMENTS},
})
breakeven

## 4 · Sensitivity: does the recommendation change with the assumptions?

In [ ]:
sens_rows = []
for margin in [0.2, 0.3, 0.5]:
    for cost in [0.0, 0.01, 0.05, 0.10]:
        vals = {name: pol.policy_value(df, p, margin, cost, CONTROL)[0] for name, p in policies.items()
                if name != "Learned rule (in-sample)"}
        best = max(vals, key=vals.get)
        sens_rows.append({"margin": margin, "cost_per_email": cost, "best_rule": best,
                          "profit_per_1000_best": vals[best] * 1000,
                          "mens_to_all_vs_no_email_per_1000": (vals["Men's e-mail to all"] - vals["No e-mail"]) * 1000})
pd.DataFrame(sens_rows)

## 5 · Recommendation

*Fill in after running — this feeds the one-page memo in `reports/`:*

- **Decision:** …
- **Expected value:** …
- **Why not targeting:** …
- **What we could not measure:** unsubscribes, long-term effects beyond two weeks, and e-mail fatigue were not in the data; a follow-up test should track unsubscribe rate as a guardrail metric.
- **Next test:** …